In [2]:
import cv2
import glob
import pandas as pd
import numpy as np

# ==========================================
# 1. THE BASELINE ALGORITHM (Part b)
# ==========================================
def find_lane_centers_baseline(image):
    threshold_value = 200
    expected_width_near = 258  
    expected_width_far = 163   
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY)
    
    def process_row(row_pixels, expected_width):
        white_indices = np.where(row_pixels == 255)[0]
        left_pixels = white_indices[white_indices < 240]
        right_pixels = white_indices[white_indices >= 240]
        
        u_left = int(np.mean(left_pixels)) if len(left_pixels) > 0 else None
        u_right = int(np.mean(right_pixels)) if len(right_pixels) > 0 else None
            
        if u_left is not None and u_right is not None:
            return int((u_left + u_right) / 2)
        elif u_left is not None:
            return int(u_left + (expected_width / 2))
        elif u_right is not None:
            return int(u_right - (expected_width / 2))
        else:
            return None

    u_n = process_row(thresh[260, :], expected_width_near)
    u_f = process_row(thresh[170, :], expected_width_far)
    return u_n, u_f

# ==========================================
# 2. THE REPAIRED ALGORITHM (Part d)
# ==========================================
class RobustLaneTracker:
    def __init__(self):
        self.W_n = 258  
        self.W_f = 163  
        self.prev_un = 240
        self.prev_uf = 240
        self.search_radius = 40

    def process_row(self, row_pixels, expected_width, prev_center):
        # ROI Prediction Window
        expected_left = prev_center - (expected_width // 2)
        expected_right = prev_center + (expected_width // 2)
        
        left_min = max(0, expected_left - self.search_radius)
        left_max = min(479, expected_left + self.search_radius)
        right_min = max(0, expected_right - self.search_radius)
        right_max = min(479, expected_right + self.search_radius)

        white_indices = np.where(row_pixels == 255)[0]
        
        # Apply strict Search ROI Blinders
        left_pixels = white_indices[(white_indices >= left_min) & (white_indices <= left_max)]
        right_pixels = white_indices[(white_indices >= right_min) & (white_indices <= right_max)]
        
        u_left = int(np.mean(left_pixels)) if len(left_pixels) > 0 else None
        u_right = int(np.mean(right_pixels)) if len(right_pixels) > 0 else None

        current_center = prev_center 
        confidence = 0.0
        
        if u_left is not None and u_right is not None:
            current_center = int((u_left + u_right) / 2)
            confidence = 1.0 if abs((u_right - u_left) - expected_width) < 20 else 0.8
        elif u_left is not None:
            current_center = int(u_left + (expected_width / 2))
            confidence = 0.5 
        elif u_right is not None:
            current_center = int(u_right - (expected_width / 2))
            confidence = 0.5 
        else:
            current_center = prev_center
            confidence = 0.1 
            
        return current_center, confidence

    def find_centers(self, image):
        # Lower threshold to see inside shadows
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY)
        
        self.prev_un, conf_n = self.process_row(thresh[260, :], self.W_n, self.prev_un)
        self.prev_uf, conf_f = self.process_row(thresh[170, :], self.W_f, self.prev_uf)
            
        return self.prev_un, self.prev_uf, conf_n, conf_f

# ==========================================
# 3. GRADING LOGIC (Part e)
# ==========================================
def evaluate_performance(predictions_dict, df_truth, sequence_name, row_target):
    errors = []
    unknown_count = 0
    total_frames = 24 
    
    df_seq = df_truth[df_truth['sequence'] == sequence_name]
    
    for index, row in df_seq.iterrows():
        frame_id = int(row['frame'])
        true_val = row[row_target]
        pred_val = predictions_dict.get(frame_id, None)
        
        if pred_val is None or pd.isna(true_val):
            unknown_count += 1
        else:
            errors.append(abs(pred_val - true_val))
            
    mae = np.mean(errors) if len(errors) > 0 else 0
    unknown_rate = (unknown_count / total_frames) * 100
    
    return mae, unknown_rate

# ==========================================
# 4. EXECUTION LOOP (ALL SEQUENCES)
# ==========================================
# Update these paths to match your local computer exactly
base_lane_path = r"C:/Users/vinay/OneDrive - Indian Institute of Technology Bhubaneswar/Desktop/sedrica assignment/city/lane"
csv_path = r"C:/Users/vinay/OneDrive - Indian Institute of Technology Bhubaneswar/Desktop/sedrica assignment/city/lane_reference.csv"

# Load Ground Truth Answer Key
df_truth = pd.read_csv(csv_path)

# Initialize the repaired tracker
tracker = RobustLaneTracker()

# Test all three sequences for the final report table
sequences_to_test = ['clear', 'missing', 'shadow']

for seq in sequences_to_test:
    print(f"\n======================================================")
    print(f"       PERFORMANCE ON '{seq.upper()}' SEQUENCE")
    print(f"======================================================")
    
    # Reset tracker memory for each new sequence
    tracker.prev_un = 240
    tracker.prev_uf = 240
    
    # Grab all 24 frames from the specific folder
    image_paths = sorted(glob.glob(f"{base_lane_path}/{seq}/*.png"))
    
    baseline_un, baseline_uf = {}, {}
    repaired_un, repaired_uf = {}, {}
    
    for frame_idx, path in enumerate(image_paths):
        img = cv2.imread(path)
        if img is not None:
            # 1. Run Baseline
            base_un, base_uf = find_lane_centers_baseline(img)
            baseline_un[frame_idx] = base_un
            baseline_uf[frame_idx] = base_uf
            
            # 2. Run Repaired
            rep_un, rep_uf, _, _ = tracker.find_centers(img)
            repaired_un[frame_idx] = rep_un
            repaired_uf[frame_idx] = rep_uf
            
    # 3. Grade Both using exact column names from your CSV
    b_mae_un, b_unk_un = evaluate_performance(baseline_un, df_truth, seq, 'near_center_x_px')
    b_mae_uf, b_unk_uf = evaluate_performance(baseline_uf, df_truth, seq, 'far_center_x_px')
    
    r_mae_un, r_unk_un = evaluate_performance(repaired_un, df_truth, seq, 'near_center_x_px')
    r_mae_uf, r_unk_uf = evaluate_performance(repaired_uf, df_truth, seq, 'far_center_x_px')
    
    # 4. Print Results for the Report Table
    print("BASELINE (Part B):")
    print(f"  Near Row (v_n=260) -> MAE: {b_mae_un:.2f} px | Unknown Rate: {b_unk_un:.1f}%")
    print(f"  Far Row  (v_f=170) -> MAE: {b_mae_uf:.2f} px | Unknown Rate: {b_unk_uf:.1f}%")
    
    print("\nREPAIRED (Part D):")
    print(f"  Near Row (v_n=260) -> MAE: {r_mae_un:.2f} px | Unknown Rate: {r_unk_un:.1f}%")
    print(f"  Far Row  (v_f=170) -> MAE: {r_mae_uf:.2f} px | Unknown Rate: {r_unk_uf:.1f}%")


       PERFORMANCE ON 'CLEAR' SEQUENCE
BASELINE (Part B):
  Near Row (v_n=260) -> MAE: 0.60 px | Unknown Rate: 0.0%
  Far Row  (v_f=170) -> MAE: 0.79 px | Unknown Rate: 0.0%

REPAIRED (Part D):
  Near Row (v_n=260) -> MAE: 0.60 px | Unknown Rate: 0.0%
  Far Row  (v_f=170) -> MAE: 0.79 px | Unknown Rate: 0.0%

       PERFORMANCE ON 'MISSING' SEQUENCE
BASELINE (Part B):
  Near Row (v_n=260) -> MAE: 0.61 px | Unknown Rate: 0.0%
  Far Row  (v_f=170) -> MAE: 0.98 px | Unknown Rate: 37.5%

REPAIRED (Part D):
  Near Row (v_n=260) -> MAE: 0.61 px | Unknown Rate: 0.0%
  Far Row  (v_f=170) -> MAE: 19.71 px | Unknown Rate: 0.0%

       PERFORMANCE ON 'SHADOW' SEQUENCE
BASELINE (Part B):
  Near Row (v_n=260) -> MAE: 0.57 px | Unknown Rate: 0.0%
  Far Row  (v_f=170) -> MAE: 1.04 px | Unknown Rate: 0.0%

REPAIRED (Part D):
  Near Row (v_n=260) -> MAE: 0.61 px | Unknown Rate: 0.0%
  Far Row  (v_f=170) -> MAE: 0.81 px | Unknown Rate: 0.0%
